In [ ]:
# Colab bootstrap — auto-clone repo on Google Colab, no-op locally
import os, sys, subprocess

REPO = "https://github.com/jongmoonha/AI-PHM_Graduate.git"
DIR  = "AI-PHM_Graduate"

try:
    import google.colab  # type: ignore
    target = '/content/' + DIR
    if not os.path.isdir(target):
        subprocess.run(["git", "clone", REPO, target], check=True)
    os.chdir(target)
    print('Google Colab detected. Working directory:', os.getcwd())
except ImportError:
    print('Local environment detected. Working directory:', os.getcwd())


# FFT 기초 — 수식 없이 주파수 분석 시작하기

진동 신호를 시간 영역에서 관찰하는 것만으로는 어떤 주기 성분이 들어 있는지 알기 어렵다. 이 노트북에서는 다음 두 단계로 주파수 분석의 기초 감각을 익힌다.

1. **실습 1.** 시간 영역에서 신호를 로드하고 일부 구간을 확대하여 주기성을 눈으로 확인한다.
2. **실습 2.** `utils.fft` 로 단측 진폭 스펙트럼을 얻고, 50~70 Hz 구간을 확대하여 bin 해상도와 누설 (leakage) 효과를 체감한다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import utils

---

## 실습 1. 시간 영역 신호 로드 및 확대 관찰

샘플링 주파수 `fs = 1000 Hz` 로 1초 동안 취득된 진동 신호를 로드한다.
전체 구간(0~1 s)에서 한 번 그린 뒤, 0~0.2 s 구간으로 확대하여 반복되는 파형을 확인한다.

In [ ]:
fs = 1000

data = np.array(pd.read_csv('./data/data_sample_fft.csv'))

t = data[:, 1]
v = data[:, 2]

fig, ax = plt.subplots(2, 1, figsize=(10, 5))
ax[0].plot(t, v, 'C0')
ax[0].set_xlabel('Time (s)'); ax[0].set_ylabel('Amplitude')
ax[0].set_title('Full record (0 ~ 1 s)')

ax[1].plot(t, v, 'C0')
ax[1].set_xlim([0, 0.2])
ax[1].set_xlabel('Time (s)'); ax[1].set_ylabel('Amplitude')
ax[1].set_title('Zoom-in (0 ~ 0.2 s)')

fig.tight_layout(); plt.show()

**관찰.** 전체 1초 구간에서는 값이 빠르게 오르내려 잡음처럼 보이지만,
0~0.2 s 구간으로 확대하면 약 60 Hz (주기 ≈ 1/60 s ≈ 16.7 ms) 의 주기 성분이 뚜렷이 관찰된다.
즉 신호는 "무질서"한 것이 아니라, 짧은 창으로 보면 구조가 드러난다.
이것이 주파수 분석이 필요한 첫 번째 이유다 — 시간 영역 관찰만으로는 성분의 개수·진폭을 정량화하기 어렵다.

## 실습 2. `utils.fft` 로 단측 스펙트럼 + 50~70 Hz 확대

`utils.fft(v, fs)` 는 실수 신호에 대해 단측 진폭 스펙트럼 `(f, A)` 를 반환한다. 전체 스펙트럼 (plot) 과 60 Hz 주변 (50~70 Hz) 확대 (stem) 를 함께 그려 bin 간격 `fs/N` 과 이웃 bin 으로의 누설을 관찰한다.

In [ ]:
f, A_util = utils.fft(v, fs)
N = len(v)
print('bin spacing fs/N = ', fs / N, 'Hz')

fig, ax = plt.subplots(3, 1, figsize=(10, 9))

ax[0].plot(t, v, 'C0')
ax[0].set_xlabel('Time (s)'); ax[0].set_ylabel('Amplitude')
ax[0].set_title('Time Domain')

# 포인트 수가 많을 때는 plot 이 가독성이 좋다
ax[1].plot(f, A_util, 'C1')
ax[1].set_xlabel('Frequency (Hz)'); ax[1].set_ylabel('Amplitude')
ax[1].set_title('Frequency Domain (full)')

# 확대 구간은 개별 bin 이 보이도록 stem 이 적합
ax[2].stem(f, A_util, linefmt='C1-', markerfmt='C1o', basefmt=' ')
ax[2].set_xlim([50, 70])
ax[2].set_xlabel('Frequency (Hz)'); ax[2].set_ylabel('Amplitude')
ax[2].set_title('Frequency Domain (zoom 50 ~ 70 Hz)')

fig.tight_layout(); plt.show()

**관찰.** 전체 스펙트럼에서는 60 Hz 한 지점만 눈에 띄지만, 50~70 Hz 로 확대하면 bin 간격이 `fs/N = 1000/1000 = 1 Hz` 임을 확인할 수 있고, 60 Hz 옆 bin (59, 61 Hz) 에도 작은 진폭이 남는 누설 (spectral leakage) 을 볼 수 있다. 이 누설은 유한 관측 구간의 불가피한 효과로, 다음 노트북들에서 윈도우·필터링으로 완화하는 방법을 다룬다.

---